In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable

from torch.utils.data import Dataset,DataLoader

from tokenizers import ByteLevelBPETokenizer
from tokenizers.trainers import BpeTrainer
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers, processors
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace

from torchinfo import summary
import math


САМ МИПЛ

In [10]:
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)


tokenizer.decoder = decoders.ByteLevel()
trainer = BpeTrainer(
    vocab_size=15000,
    special_tokens=["[PAD]", "[UNK]", "[SOS]", "[EOS]"],
    min_frequency=2
)

files = ["VocabText.txt"]
tokenizer.train(files, trainer)

sos_token_id = tokenizer.token_to_id("[SOS]")
eos_token_id = tokenizer.token_to_id("[EOS]")

MAX_LEN = 512
tokenizer.enable_padding(length=MAX_LEN)
tokenizer.enable_truncation(max_length=MAX_LEN)
tokenizer.save("miple_tokenizer.json")



In [11]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        self.qkv = nn.Linear(embed_size, embed_size * 3)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        qkv = qkv.reshape(B, T, 3, self.heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(T, T)).to(x.device)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.fc_out(out)

In [12]:
class Block(nn.Module):
    def __init__(self, embed_size, heads, dropout=0.1):
        super().__init__()
        self.attn = SelfAttention(embed_size, heads)
        self.ln1 = nn.LayerNorm(embed_size)
        self.ff = nn.Sequential(
            nn.Linear(embed_size, 4 * embed_size),
            nn.GELU(),
            nn.Linear(4 * embed_size, embed_size),
            nn.Dropout(dropout),
        )
        self.ln2 = nn.LayerNorm(embed_size)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [13]:
class MipleModel(nn.Module):
    def __init__(self, vocab_size, embed_size, block_size, heads, decisions_count, emotions_count, market_features, n_layers, dropout=0.1):
        super(MipleModel, self).__init__()
        self.block_size = block_size
        self.token_embed = nn.Embedding(vocab_size, embed_size)
        self.pos_embed = nn.Embedding(block_size, embed_size)

        self.blocks = nn.Sequential(*[Block(embed_size, heads, dropout) for _ in range(n_layers)])

        self.market_proj = nn.Linear(market_features, embed_size)
        
        combined_input_size = (embed_size * 3) + emotions_count

        self.buy_decision = nn.Sequential(
            nn.Linear(combined_input_size, embed_size),
            nn.ReLU(),
            nn.Linear(embed_size, embed_size),
            nn.Linear(embed_size, decisions_count),
        )

        self.emotions_choicer = nn.Linear(combined_input_size, emotions_count)

        self.ln_f = nn.LayerNorm(embed_size)
        self.fc = nn.Linear(embed_size, vocab_size)

    def forward(self, market_seq, prompt_seq, news_seq, current_emotions):
        B, T = prompt_seq.shape
        

        token_embeddings = self.token_embed(prompt_seq)
        position_embeddings = self.pos_embed(torch.arange(T, device=prompt_seq.device))
        x = token_embeddings + position_embeddings
        x = self.blocks(x)


        market_emb = self.market_proj(market_seq)
        news_emb = self.token_embed(news_seq) 

        market_emb_mean = torch.mean(market_emb, dim=1)
        news_emb_mean = torch.mean(news_emb, dim=1)
        
        x_mean = torch.mean(x, dim=1)

        combined = torch.cat((x_mean, market_emb_mean, news_emb_mean, current_emotions), dim=1)

        trade_logits = self.buy_decision(combined)
        new_emotions = self.emotions_choicer(combined)


        x_norm = self.ln_f(x)
        text_logits = self.fc(x_norm) 

        return trade_logits, new_emotions, text_logits

In [14]:
# тестовый прогон
vocab_size = tokenizer.get_vocab_size()
embed_size = 512
block_size = 512
heads = 8
decisions_count = 3
emotions_count = 8
market_features = 50
n_layers = 4

model = MipleModel(vocab_size, embed_size, block_size, heads, decisions_count, emotions_count, market_features, n_layers)

dummy_prompt = torch.randint(0, vocab_size, (2, 128), dtype=torch.long)
dummy_market = torch.randn(2, 50, market_features)
dummy_news = torch.randint(0, vocab_size, (2, 20), dtype=torch.long)
dummy_emotions = torch.randn(2, emotions_count)


trade_logits, new_emotions, text_logits = model(dummy_market,dummy_prompt, dummy_news, dummy_emotions)

print("trade logits shape:", trade_logits.shape)
print("new emotions shape:", new_emotions.shape)
print("text logits shape:", text_logits.shape)

model_stats = summary(
    model, 
    input_data=(dummy_market, dummy_prompt, dummy_news, dummy_emotions),
    col_names=["input_size", "output_size", "num_params", "kernel_size"],
    depth=3
)

print(model_stats)

trade logits shape: torch.Size([2, 3])
new emotions shape: torch.Size([2, 8])
text logits shape: torch.Size([2, 128, 3261])
Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Kernel Shape
MipleModel                               [2, 50, 50]               [2, 3]                    --                        --
├─Embedding: 1-1                         [2, 128]                  [2, 128, 512]             1,669,632                 --
├─Embedding: 1-2                         [128]                     [128, 512]                262,144                   --
├─Sequential: 1-3                        [2, 128, 512]             [2, 128, 512]             --                        --
│    └─Block: 2-1                        [2, 128, 512]             [2, 128, 512]             --                        --
│    │    └─LayerNorm: 3-1               [2, 128, 512]             [2, 128, 512]             1,024                     --
│    │    └─

In [15]:
class FineTuningDataset(Dataset):
    def __init__(self, tokenizer, history, max_len=512):
        super().__init__()
        self.tokenizer = tokenizer
        self.history = history

        self.stocks_data = history['stocks_states']
        self.messages_data = history['messages']
        self.events = history['events']
        self.emotions_data = history['emotions']

        self.max_len = max_len
        self.chat_samples = []

        for i in range(len(self.messages_data)):
            text = f"{self.messages_data[i]['input']} [SOS] {self.messages_data[i]['output']} [EOS]"
            self.chat_samples.append(text)

    def __len__(self):
        return len(self.chat_samples)
            
    def __getitem__(self, index):
        encoded_messages = self.tokenizer.encode(self.chat_samples[index]).ids
        encoded_events = self.tokenizer.encode(self.events[index]).ids

        text_out_ids = torch.tensor(encoded_messages, dtype=torch.long)
        events_out = torch.tensor(encoded_events, dtype=torch.long)

        stocks_out = torch.tensor(self.stocks_data[index], dtype=torch.float)
        emotions_out = torch.tensor(self.emotions_data[index], dtype=torch.float)

        return text_out_ids, events_out, stocks_out, emotions_out

In [16]:
import math
import random
import json
import pickle
import os

class AffectionBlock:
    def __init__(self, idx, entered_blocks = [], output_blocks = [], base_affection = 0, epsilon = 1e-5):
        self.entered_blocks = entered_blocks
        self.output_blocks = output_blocks
        self.idx = idx
        self.epsilon = epsilon

        self.base_affection = base_affection
        self.affection = len(self.entered_blocks) + base_affection
    
    def to_affect(self):
        affects = {}
        if self.affection == 0:
            return affects
        for block in self.entered_blocks:
            denom = block.affection if block.affection > self.epsilon else self.epsilon
            affect = math.fabs(self.affection - block.affection) * (self.affection / denom)
            affects[block.idx] = affect
        return affects

    def to_forgetting(self, current_idx, forgetting_step = 0.005):
        mem_age = current_idx - self.idx
        if mem_age > 0:
            forget_rate = math.exp(-forgetting_step)
            self.base_affection = forget_rate * self.base_affection
            self.affection = self.base_affection + len(self.entered_blocks)

class AffectorText(AffectionBlock):
    def __init__(self, idx, entered_blocks=[], output_blocks=[], base_affection=0, epsilon=0.0001):
        super().__init__(idx, entered_blocks, output_blocks, base_affection, epsilon)
        
        self.w_2idx = {}
        self.word_blocks = {}
        
        self.sentiment_modifiers = {
        "оптимист":      {"boost": 2.0, "forget_speed": 0.0002},
        "рискованный":    {"boost": 2.5, "forget_speed": 0.0001},
        "систематичный":  {"boost": 1.0, "forget_speed": 0.001},
        "консерватист":  {"boost": 0.8, "forget_speed": 0.0015},
        "интроверт":      {"boost": 0.5, "forget_speed": 0.003},
        "убежденный":     {"boost": 1.2, "forget_speed": 0.0005},
        "интуитивный":    {"boost": 1.8, "forget_speed": 0.0004}
        }
        
        self.cfg_sentiments = {
        "рискованный":    {"temperature": 1.5, "forget_speed": 0.0001, "loop_penalty": 0.2}, 
        "оптимист":       {"temperature": 0.9, "forget_speed": 0.0002, "loop_penalty": 0.5},
        "систематичный":  {"temperature": 0.3, "forget_speed": 0.001,  "loop_penalty": 2.0},
        "консерватист":  {"temperature": 0.2, "forget_speed": 0.0015, "loop_penalty": 2.5},
        "интроверт":      {"temperature": 0.4, "forget_speed": 0.003,  "loop_penalty": 1.5},
        "убежденный":     {"temperature": 0.1, "forget_speed": 0.0005, "loop_penalty": 3.0},
        "интуитивный":    {"temperature": 1.1, "forget_speed": 0.0004, "loop_penalty": 0.7}
        }
            
    def train_on_text(self, json_file_path, word_blocks, idx_to_word, unique_words):
            with open(json_file_path, 'r', encoding='utf-8') as f:
                dataset = json.load(f)
                
            total_tokens = 0
            for data_item in dataset:
                replics = data_item.get("replics", [])
                news = data_item.get("news", [])
                answer = data_item.get("answer", "")
                sentiments = data_item.get("sentiments", ["|"])
                
                current_sentiment = sentiments[0]
                modifier = self.sentiment_modifiers.get(current_sentiment, {"boost": 1, "forget_speed": 0.0001})
                
                segments = [
                    " ".join(replics).split(),
                    "".join(news).split(),
                    answer.split()
                ]

                for tokens in segments:
                    if len(tokens) < 2: 
                        continue
                    total_tokens += len(tokens)
                    for i in range(len(tokens) - 1):
                        curr_w = tokens[i]
                        next_w = tokens[i+1]
                        for w in (curr_w, next_w):
                            if w not in word_blocks:
                                new_idx = len(word_blocks)
                                initial_affection = 1 * modifier["boost"]
                                word_blocks[w] = AffectionBlock(
                                    idx=new_idx, 
                                    entered_blocks=[], 
                                    output_blocks=[], 
                                    base_affection=initial_affection
                                )
                                idx_to_word[new_idx] = w
                        
                        curr_block = word_blocks[curr_w]
                        next_block = word_blocks[next_w]
                        
                        if curr_block not in next_block.entered_blocks:
                            next_block.entered_blocks.append(curr_block)
                        if next_block not in curr_block.entered_blocks:
                            curr_block.output_blocks.append(next_block)
                            
                        next_block.base_affection += 0.25 * modifier["boost"]
                        
                    for block in word_blocks.values():
                        block.to_forgetting(len(tokens), forgetting_step=modifier["forget_speed"])

            for block in word_blocks.values():
                block.affection = len(block.entered_blocks) + block.base_affection
                
            print(f'trained on: {total_tokens} samples')
        
    def generate(self, prompt, current_mood, word_blocks, idx_2w, max_len=25):
        cfg = self.cfg_sentiments.get(current_mood, {"temperature": 0.5, "forget_speed": 0.001, "loop_penalty": 1.0})
        
        input_words = prompt.lower().split()
        start_block = None
        
        for w in input_words:
            clean_w = w.strip("?,.!;\"'")
            if not clean_w: continue
            
            found_nodes = [block for word, block in word_blocks.items() if clean_w == word.lower()]
            if not found_nodes:
                 found_nodes = [block for word, block in word_blocks.items() if clean_w in word.lower()]
                 
            if found_nodes:
                start_block = random.choice(found_nodes)
                break
            
        if not start_block:
            if word_blocks:
                start_block = random.choice(list(word_blocks.values()))
            else:
                return "Нет данных для генерации."
            
        current_block = start_block
        generated_words = [idx_2w[current_block.idx]]
        
        visited_counter = {block.idx: 0 for block in word_blocks.values()}
        visited_counter[current_block.idx] += 1
        
        current_step = 0
        
        while len(generated_words) < max_len:
            current_step += 1
            
            for block in word_blocks.values():
                block.to_forgetting(current_step, forgetting_step=cfg["forget_speed"])
                
            candidates = current_block.output_blocks
            if not candidates:
                break
                
            weights = []
            for cand in candidates:
                affects = cand.to_affect()

                base_weight = affects.get(current_block.idx, cand.epsilon)
                base_weight += cand.epsilon * cfg["temperature"]

                penalty = math.exp(-cfg["loop_penalty"] * visited_counter[cand.idx])
                weights.append(base_weight * penalty)
                
            if sum(weights) == 0:
                weights = [1.0] * len(candidates)
                
            next_block = random.choices(candidates, weights=weights, k=1)[0]
            

            next_block.base_affection += 0.5
            next_block.affection = len(next_block.entered_blocks) + next_block.base_affection
            
            word_str = idx_2w[next_block.idx]
            generated_words.append(word_str)
            
            visited_counter[next_block.idx] += 1
            current_block = next_block
            
            if word_str.endswith('.') or word_str.endswith('!') or word_str.endswith('?') or '[EOS]' in word_str:
                break
        
        answer_sentence = " ".join(generated_words)
        

        clean_answer = answer_sentence.replace('[EOS]', '').replace('[SOS]', '').replace('[NEWS]', '')
        
        return clean_answer.strip().capitalize()
    
    def generate_with_confidence(self, prompt, current_mood, word_blocks, idx_2w, max_len=20):
            cfg = self.cfg_sentiments.get(current_mood, {"temperature": 0.5, "forget_speed": 0.001, "loop_penalty": 1.5})
            
            input_words = prompt.lower().split()
            

            is_echo_context = len(input_words) > 2 and len(set(input_words)) <= 3
            
            current_temp = 2.0 if is_echo_context else cfg["temperature"]

            stop_words = {
                "что", "как", "где", "когда", "почему", "в", "на", "с", "из", "по", "а", "но", "и", 
                "не", "ни", "это", "этот", "эта", "видели", "знаете", "вы", "мы", "они", "ты", "я"
            }

            context_blocks = []
            for w in input_words:
                clean_w = w.strip("?,.!;\"'() ")
                if not clean_w or clean_w in stop_words:
                    continue
                found = [block for word, block in word_blocks.items() if clean_w == word.lower()]
                context_blocks.extend(found)

            start_block = None
            
            if is_echo_context:
                meaningful_blocks = [b for w, b in word_blocks.items() if w.lower() not in stop_words and w.lower() not in input_words]
                if meaningful_blocks:
                    start_block = random.choice(meaningful_blocks)

            if not start_block:
                for w in reversed(input_words):
                    clean_w = w.strip("?,.!;\"'() ")
                    if not clean_w or clean_w in stop_words: 
                        continue
                    found_nodes = [block for word, block in word_blocks.items() if clean_w == word.lower()]
                    if found_nodes:
                        start_block = random.choice(found_nodes)
                        break

            if not start_block and word_blocks:
                meaningful_blocks = [b for w, b in word_blocks.items() if w.lower() not in stop_words]
                if meaningful_blocks:
                    start_block = random.choice(meaningful_blocks)
                else:
                    start_block = random.choice(list(word_blocks.values()))
            elif not start_block:
                return "...", 10.0
            
            current_block = start_block
            generated_words = [idx_2w[current_block.idx]]
            
            local_visited = {block.idx: 0 for block in word_blocks.values()}
            local_visited[current_block.idx] += 1
            
            confidence_scores = []
            current_step = 0
            
            while len(generated_words) < max_len:
                current_step += 1
                
                for block in word_blocks.values():
                    block.to_forgetting(current_step, forgetting_step=cfg["forget_speed"])
                    
                candidates = current_block.output_blocks
                if not candidates:
                    break
                    
                weights = []
                for cand in candidates:
                    cand_word = idx_2w[cand.idx].lower()
                    affects = cand.to_affect()
                    
                    base_weight = affects.get(current_block.idx, cand.epsilon)
                    # Применяем измененную температуру
                    base_weight += cand.epsilon * current_temp
                    
                    context_bonus = 0.0
                    for ctx_block in context_blocks:
                        if ctx_block.idx in affects:
                            context_bonus += affects[ctx_block.idx] * 2.5
                    
                    if len(generated_words) > 1:
                        prev_word_str = generated_words[-2]
                        prev_block = word_blocks.get(prev_word_str)
                        if prev_block and prev_block.idx in affects:
                            context_bonus += affects[prev_block.idx] * 1.5

                    base_weight += context_bonus

                    penalty = 1.0
                    if local_visited[cand.idx] > 0:
                        if cand_word in stop_words:
                            penalty = 0.001
                        else:
                            penalty = pow(0.3, local_visited[cand.idx] * cfg["loop_penalty"])
                    
                    if cand_word in input_words:
                        penalty *= 0.1
                            
                    weights.append(base_weight * penalty)
                    
                total_w = sum(weights)
                if total_w <= 0:
                    weights = [affects.get(current_block.idx, cand.epsilon) for cand in candidates]
                    if sum(weights) == 0: break
                    
                if total_w > 0:
                    confidence_scores.append(max(weights) / (total_w + 1e-9))
                    
                next_block = random.choices(candidates, weights=weights, k=1)[0]
                next_block.base_affection += 0.02 
                
                word_str = idx_2w[next_block.idx]
                generated_words.append(word_str)
                local_visited[next_block.idx] += 1
                current_block = next_block
                
                if word_str.endswith('.') or word_str.endswith('!') or word_str.endswith('?'):
                    break
            
            answer_sentence = " ".join(generated_words)
            
            avg_confidence = sum(confidence_scores) / len(confidence_scores) if confidence_scores else 0.5
            if is_echo_context:
                confidence_pct = random.randint(15, 35)
            else:
                confidence_pct = min(99, max(15, int(avg_confidence * 100)))
            
            if answer_sentence.lower().strip("?.!") == prompt.lower().strip("?.!"):
                topics = ["Ну допустим.", "И что с того?", "Ладно, забей.", "А смысл?", "Погнали дальше."]
                return random.choice(topics), 20
            
            return answer_sentence.strip().capitalize(), confidence_pct
    
    
    def save(self, word_blocks, idx_2w, filename="aff.pkl"):
        data = { "word_blocks": word_blocks, "idx_2w": idx_2w}    
        with open(filename, 'wb') as f:
            pickle.dump(data, f)
        print(f"model was saved as {filename}")
        
    def load(self, filename="mirpus_model.pkl"):
        if not os.path.exists(filename):
            return None
        try:
            with open(filename, 'rb') as f:
                data = pickle.load(f)
            print(f"succesful {filename}")
            return data["word_blocks"], data["idx_2w"]
        except Exception as e:
            print(f"err: {e}")
            return None
        